<a href="https://colab.research.google.com/github/DulaniDeSilva/Natural-Language-Processing-NLP-/blob/main/SentimentAnalysis_Lect1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sentiment Analysis using Python


kaggle: Sentiment140 dataset
https://www.kaggle.com/datasets/kazanova/sentiment140

About the dataset: There are 1.6 million tweets used for sentiment analysis
Each tweet is labeled as 0 for negative and 4 for positive.

It comes from twitter data(2009)

Format
[0] sentiment label (0 or 4)
[1] tweet ID
[2] date
[3] query (unused)
[4] username
[5] tweet text

---



## Mount the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 1: Unzip the dataset if you have uploaded it to the drive

In [ ]:
zip_path = "/content/drive/MyDrive/NIBM_LECTURE/NLP/PT/Lecture1/archive.zip"

Unzip the file

In [ ]:
import zipfile

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/drive/MyDrive/NIBM_LECTURE/NLP/PT/Lecture1/dataset")

print("Dataset extracted!")

Dataset extracted!


Step 2: Load the dataset

In [ ]:
import pandas as pd
file_path = "/content/drive/MyDrive/NIBM_LECTURE/NLP/PT/Lecture1/dataset/training.1600000.processed.noemoticon.csv"

df = pd.read_csv(file_path, encoding='latin-1', header=None)

# file type = csv
# Encoding - latin-1 why? tweets contain emojis, special characters, so if we used utf-8 it may fail, latin-1 avoids decoding erros.

# Explore data

df.head() - sample rows from the begining
df.size - shape
df.info - data types
df.tail() - last

View first few rows

In [ ]:
df.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In column 0 => What we need to predict (output/ label)
  o => negative tweet
  4 => positive tweet


Column 5 contains the text of the tweet => input text (what model reads)


Other columns (1,2,3, 4) are extra information like tweetID which we do not need for basic sentiment analysis.

In [ ]:
df.head(10)

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
5,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew
6,0,1467811592,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,mybirch,Need a hug
7,0,1467811594,Mon Apr 06 22:20:03 PDT 2009,NO_QUERY,coZZ,@LOLTrish hey long time no see! Yes.. Rains a...
8,0,1467811795,Mon Apr 06 22:20:05 PDT 2009,NO_QUERY,2Hood4Hollywood,@Tatiana_K nope they didn't have it
9,0,1467812025,Mon Apr 06 22:20:09 PDT 2009,NO_QUERY,mimismo,@twittera que me muera ?


In [ ]:
df.tail()

,0,1,2,3,4,5
1599995,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1599996,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1599999,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [ ]:
print("Shape/ size of the dataset is: ", df.shape)

Shape/ size of the dataset is:  (1600000, 6)


Viewing the column information

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   0       1600000 non-null  int64 
 1   1       1600000 non-null  int64 
 2   2       1600000 non-null  object
 3   3       1600000 non-null  object
 4   4       1600000 non-null  object
 5   5       1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB


Since we are interested in column 0 and column 5 we will keep these two columns

In [ ]:
df_new = df[[0, 5]]
df_new.columns = ["label", "text"]
# df_new.head()
df_new.tail()

,label,text
1599995,4,Just woke up. Having no school is the best fee...
1599996,4,TheWDB.com - Very cool to hear old Walt interv...
1599997,4,Are you ready for your MoJo Makeover? Ask me f...
1599998,4,Happy 38th Birthday to my boo of alll time!!! ...
1599999,4,happy #charitytuesday @theNSPCC @SparksCharity...


Convert labels 4 -> 1

Why: Original labels are 0 for negative, 1 for positive

machine learning prefers 0 for negative and 1 for positive

Make it easy for binary classification

In [ ]:
df_new["label"] = df_new["label"].replace(4, 1)
df_new.tail()

/tmp/ipykernel_2076/2925063396.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new["label"] = df_new["label"].replace(4, 1)


,label,text
1599995,1,Just woke up. Having no school is the best fee...
1599996,1,TheWDB.com - Very cool to hear old Walt interv...
1599997,1,Are you ready for your MoJo Makeover? Ask me f...
1599998,1,Happy 38th Birthday to my boo of alll time!!! ...
1599999,1,happy #charitytuesday @theNSPCC @SparksCharity...


Step 3: Explore the data

In [ ]:
print("First 5 rows:")
print(df_new.head())

print("\nDataset Info:")
print(df_new.info())

print("\nLabel Distribution:")
print(df_new["label"].value_counts())



First 5 rows:
   label                                               text
0      0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1      0  is upset that he can't update his Facebook by ...
2      0  @Kenichan I dived many times for the ball. Man...
3      0    my whole body feels itchy and like its on fire 
4      0  @nationwideclass no, it's not behaving at all....

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 2 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   label   1600000 non-null  int64 
 1   text    1600000 non-null  object
dtypes: int64(1), object(1)
memory usage: 24.4+ MB
None

Label Distribution:
label
0    800000
1    800000
Name: count, dtype: int64


Normally we have to do the preprocessing of data, here we are doing only the column selection, label coversion. But

We usually do: remove URLs, remove @mentions, remove punctuation, lowercase(not always Cook, cook), remove stopwords

Step 4: Reduce dataset size, for easy and fast training

In [ ]:
df_reduced = df_new.sample(10000, random_state=42)
print("\nReduced dataset size: ", len(df_reduced))


Reduced dataset size:  10000


Step 5: Split data

In [ ]:
from sklearn.model_selection import train_test_split    #imports train_test_split from scikit-learn (used to split your dataset into trainig adn testing parts)
X = df_reduced["text"]    # input data(features here tweet text) - what the model reads
y = df_reduced["label"]   # output (target/label) whether sentiment is negative 0 or positive 1 - what the model predicts

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)   #test_size =0.2(20% for testing, 80% for training) 10,000 tweets 8000 training, 2000 testing
#random_state = 42 controls randomness (this ensures that same split every time you run the code, without it you will get differnt train/test splits)

Step 6: Convert text to numbers

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer   #using BoW (igore grammer , and only counting how many times each word appear)

vectorizer = CountVectorizer(max_features=5000)     #creating a vocabulary of top 5000 words, removes less frequent words

X_train_vectors = vectorizer.fit_transform(X_train)   #fit - learn voab from training data, transform - convert text to number vectors
X_test_vectors = vectorizer.transform(X_test)  #only transform not fit, if we fit you may leak info from test data, model evalution becomes invalid

#X_train_vectors this is not a normal array - sparse matrxi

Step 7: Train the model

In [ ]:
from sklearn.linear_model import LogisticRegression   #use the model LogisticRegression, why: simple fast good for text classification

model = LogisticRegression(max_iter=100)
model.fit(X_train_vectors, y_train)

LogisticRegression()

Step 8: Predict

In [ ]:
y_pred = model.predict(X_test_vectors)

Step 9: Evaluate

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Accuracy: 0.7405

Confusion Matrix:
[[694 286]
 [233 787]]


Out of all predictions, 74.05% were correct

Step 10: Test custom senstences

In [ ]:
test_sentences = [
    "I love this product",
    "This is terrible",
    "Not bad at all",
    "I am very happy",
    "Worst experience ever"
]

test_vectors = vectorizer.transform(test_sentences)
predictions = model.predict(test_vectors)

for sentence, pred in zip(test_sentences, predictions):
    sentiment = "Positive" if pred == 1 else "Negative"
    print(f"{sentence} --> {sentiment}")

I love this product --> Positive
This is terrible --> Negative
Not bad at all --> Negative
I am very happy --> Positive
Worst experience ever --> Negative
